In [ ]:
# ============================================================
# SIÊU SCRIPT CÀO 200 ĐỊA DANH QUẬN 1 ĐA DẠNG (V4)
# Tự động: Quét OSM -> Việt hóa -> Tải ảnh -> Sinh AI Vector -> Đóng gói ZIP
# ============================================================

!pip install -q sentence-transformers requests icrawler

import json
import os
import shutil
import requests
import time
from icrawler.builtin import BingImageCrawler
from sentence_transformers import SentenceTransformer
from PIL import Image
from google.colab import files

# --- 1. CẤU HÌNH & VIÊT HÓA CATEGORY ---
CATEGORY_MAP = {
    "restaurant": "Nhà hàng",
    "cafe": "Quán cà phê",
    "pub": "Quán pub",
    "bar": "Quán bar",
    "museum": "Bảo tàng",
    "attraction": "Điểm tham quan",
    "viewpoint": "Điểm ngắm cảnh",
    "gallery": "Triển lãm nghệ thuật",
    "art_centre": "Trung tâm nghệ thuật",
    "place_of_worship": "Địa điểm văn hóa tâm linh",
    "theatre": "Nhà hát",
    "cinema": "Rạp chiếu phim",
    "monument": "Tượng đài/Di tích",
    "memorial": "Khu tưởng niệm",
    "park": "Công viên",
    "garden": "Vườn hoa/Cảnh quan",
    "mall": "Trung tâm thương mại",
    "historic": "Di tích lịch sử",
    "Place": "Địa điểm tham quan",
}

# Khởi tạo thư mục
if os.path.exists("media"): shutil.rmtree("media")
os.makedirs("media/locations", exist_ok=True)

# --- 2. QUÉT OSM VỚI QUERY ĐA DẠNG (MỤC TIÊU 200 ĐIỂM) ---
def fetch_200_pois():
    print("🚀 Đang quét OpenStreetMap Quận 1...")
    overpass_url = "http://overpass-api.de/api/interpreter"
    query = """
    [out:json][timeout:60];
    area["name"="Quận 1"]->.searchArea;
    (
      node["tourism"~"attraction|museum|viewpoint|gallery|art_centre"](area.searchArea);
      node["amenity"~"place_of_worship|theatre|cinema|arts_centre"](area.searchArea);
      node["historic"~"monument|memorial|statue"](area.searchArea);
      node["leisure"~"park|garden"](area.searchArea);
      node["shop"="mall"](area.searchArea);
      node["amenity"~"cafe|restaurant|pub|bar"](area.searchArea);
    );
    out body 250;
    """
    try:
        response = requests.get(overpass_url, params={'data': query}, timeout=30)
        elements = response.json().get('elements', [])
        results = []
        for item in elements:
            tags = item.get('tags', {})
            if 'name' not in tags: continue

            category = "Địa danh"
            tag_str = " ".join([str(v) for v in tags.values()]).lower()
            for eng, vie in CATEGORY_MAP.items():
                if eng.lower() in tag_str:
                    category = vie
                    break

            results.append({
                "poi_id": f"osm_{item['id']}",
                "name": tags.get('name'),
                "latitude": item['lat'],
                "longitude": item['lon'],
                "category": category,
                "address": tags.get('addr:full') or f"{tags.get('addr:street', 'Quận 1')}, TP. HCM",
                "description": f"Một {category} đặc sắc tọa lạc tại trung tâm Quận 1."
            })
        print(f"✅ Tìm thấy {len(results)} địa danh tiềm năng.")
        return results
    except Exception as e:
        print(f"❌ Lỗi OSM: {e}"); return []

# --- 3. TẢI ẢNH, VECTƠ HÓA & XUẤT FILE ---
poi_list = fetch_200_pois()
model = SentenceTransformer('clip-ViT-B-32')
try: model = model.cuda()
except: print("⚠️ Đang chạy trên CPU (Sẽ chậm hơn GPU).")

final_pois = []
TARGET_COUNT = 200

print(f"📸 Bắt đầu xử lý {TARGET_COUNT} địa danh...")
for poi in poi_list:
    if len(final_pois) >= TARGET_COUNT: break

    name, poi_uuid = poi['name'], poi['poi_id']
    save_dir = f"media/locations/{poi_uuid}"
    os.makedirs(save_dir, exist_ok=True)

    crawler = BingImageCrawler(storage={'root_dir': save_dir})
    crawler.crawl(keyword=f"{name} Saigon Tourist", max_num=3)

    files_list = [f for f in os.listdir(save_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if files_list:
        poi['image'] = f"media/locations/{poi_uuid}/{files_list[0]}"
        poi['image_list'] = [f"media/locations/{poi_uuid}/{f}" for f in files_list]
        try:
            with Image.open(os.path.join(save_dir, files_list[0])).convert('RGB') as img:
                poi['vector'] = model.encode(img).tolist()
            final_pois.append(poi)
            print(f"✅ [{len(final_pois)}/{TARGET_COUNT}] {name}")
        except: print(f"⚠️ Lỗi xử lý {name}")
    else: shutil.rmtree(save_dir)

# Đóng gói và tải về
with open('district1_diverse_200.json', 'w', encoding='utf-8') as f:
    json.dump(final_pois, f, ensure_ascii=False, indent=2)
shutil.make_archive('poi_media_bundle', 'zip', base_dir='media')
files.download('district1_diverse_200.json')
files.download('poi_media_bundle.zip')

print("\n✨ HOÀN THÀNH! Tải 2 file về và giải nén vào project.")
